# HOVITA — Thin Runner for Temporary Free GPU Environments (Colab / Kaggle / Lightning AI)

This notebook contains **zero duplicate business logic**. It configures environment variables for thin SQLite/local storage mode (when PostgreSQL/MinIO are absent), optionally starts GPU services (`ComfyUI` / `Ollama`), and executes the shared `hovita` codebase directly.

In [ ]:
import os, subprocess, sys

# 1. Clone / locate repo & install package
if not os.path.exists('pyproject.toml'):
    raise RuntimeError('Open this notebook inside the HOVITA repository root.')

subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', '-e', '.'])

# 2. Configure Thin Mode (SQLite + local object storage, GPU on-demand enabled)
os.environ['HOVITA_ENV'] = 'notebook'
os.environ['DATABASE_URL'] = 'sqlite:///data/notebook_hovita.db'
os.environ['REDIS_URL'] = 'memory://'
os.environ['STORAGE_BACKEND'] = 'local'
os.environ['LOCAL_STORAGE_DIR'] = 'data/notebook_storage'
os.environ['GPU_AVAILABLE'] = 'true'
os.environ['GPU_MODE'] = 'on_demand'

In [ ]:
# 3. Initialize DB Schema, Seed Single Persona & Verify Health
subprocess.check_call(['alembic', 'upgrade', 'head'])
subprocess.check_call([sys.executable, '-m', 'hovita.db.seed'])
subprocess.check_call([sys.executable, 'scripts/healthcheck.py'])

In [ ]:
# 4. Execute One Full GPU Batch Cycle (Identity + Production Pipeline)
import asyncio
from hovita.db.base import get_session_factory
from hovita.modules.bootstrap import build_default_registry
from hovita.agents.identity_studio import IdentityStudio
from hovita.scheduler.runner import InternalScheduler

async def run_notebook_gpu_batch():
    reg = build_default_registry()
    studio = IdentityStudio(reg)
    sched = InternalScheduler(reg)
    with get_session_factory()() as session:
        await studio.generate_multi_angle_identity(session, character_id='persona-001', approve=True)
        await studio.audition_and_lock_voice(session, character_id='persona-001', approve=True)
        return await sched.run_autonomous_cycle(session, character_id='persona-001')

result = asyncio.run(run_notebook_gpu_batch())
print(result)